# NLU Assignment 2 : Sanskrit-to-English Neural Machine Translation using pretrained IndicTrans2 model and fine-tuning it on given data
Name : Anjali Tandon

Roll No. : g25ait1021

In [47]:
!pip install -q nltk torch torchvision torchaudio datasets evaluate bert-score sentencepiece accelerate peft pandas numpy matplotlib tqdm transformers==4.46.3 huggingface_hub==0.26.5 IndicTransToolkit sacremoses mosestokenizer sacrebleu

# Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
import nltk
from nltk.translate.bleu_score import corpus_bleu
nltk.download("punkt", quiet=True)
import evaluate
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA Version :", torch.version.cuda)

In [ ]:
# Base directory
base_dir = '/content/drive/MyDrive'
data_dir= f'{base_dir}/Dataset_NLU'
output_dir = f'{base_dir}/best_model'

# HF Token
HF_TOKEN = ""

# Pre-trained model
model_name = 'ai4bharat/indictrans2-indic-en-dist-200M'

# Hyperparameters
max_source_length = 128
max_target_length = 96
batch_size = 16
epochs = 12
learning_rate = 2e-5
weight_decay = 0.05
num_beams = 5
gradient_accumulation_steps = 2
seed = 42
warmup_ratio = 0.06
label_smoothing = 0.1

In [80]:
# Function for seed setting
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(seed=42)

# Function to calculate nltk BLEU
def nltk_bleu(predictions, references):
    refs = [[r.split()] for r in references]
    hyps = [p.split() for p in predictions]
    return corpus_bleu(refs, hyps)

# Function to calculate hf evaluate BLEU
import evaluate
hf_bleu_metric = evaluate.load("bleu")
def evaluate_bleu_hf(predictions, references):
    results = hf_bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references])
    return results

# Loading Datasets

In [ ]:
import pandas as pd

# Loading the datasets
train_sa = pd.read_csv(f'{data_dir}/train_sa_10000.csv')
train_en = pd.read_csv(f'{data_dir}/train_en_10000.csv')
dev_sa = pd.read_csv(f'{data_dir}/dev_sa_1000.csv')
dev_en = pd.read_csv(f'{data_dir}/dev_en_1000.csv')
test_sa = pd.read_csv(f'{data_dir}/test_sa_1000.csv')
test_en = pd.read_csv(f'{data_dir}/test_en_1000.csv')

# Merge sanskrit and english datasets
train_df = train_sa.merge(train_en, on="Source_id", how="inner")
dev_df = dev_sa.merge(dev_en, on="Source_id", how="inner")
test_df = test_sa.merge(test_en, on="Source_id", how="inner")

print(train_df.shape)
print(dev_df.shape)
print(test_df.shape)

In [ ]:
train_df.dropna(inplace=True)
dev_df.dropna(inplace=True)
test_df.dropna(inplace=True)

train_df.reset_index(drop=True, inplace=True)
dev_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print("Train :", len(train_df))
print("Dev   :", len(dev_df))
print("Test  :", len(test_df))

# Drop duplicate rows
before = len(train_df)
train_df.drop_duplicates(subset=["Sentence_sa", "Sentence_en"],inplace=True)
train_df.reset_index(drop=True, inplace=True)
print(f"Removed {before-len(train_df)} duplicate pairs.")

In [ ]:
import re
devanagari = re.compile(r'[\u0900-\u097F]')
bad_mask = dev_df["Sentence_en"].astype(str).apply(lambda x: bool(devanagari.search(x)))
print(f"{bad_mask.sum()} / {len(dev_df)} dev rows have Devanagari text in the English column")
print(f"{(train_df['Sentence_en'].astype(str).apply(lambda x: bool(devanagari.search(x)))).sum()} / {len(train_df)} train rows have Devanagari text in the English column")

# Drop bad rows
def has_devanagari(text):
    return bool(devanagari.search(str(text)))

train_df = train_df[~train_df["Sentence_en"].apply(has_devanagari)].reset_index(drop=True)
dev_df   = dev_df[~dev_df["Sentence_en"].apply(has_devanagari)].reset_index(drop=True)

print("Train after cleanup:", len(train_df))
print("Dev after cleanup:", len(dev_df))

In [ ]:
train_df["src_len"] = train_df["Sentence_sa"].astype(str).str.split().str.len()
train_df["tgt_len"] = train_df["Sentence_en"].astype(str).str.split().str.len()
print(train_df[["src_len", "tgt_len"]].describe())

In [ ]:
train_df.head()

In [57]:
def normalize_quotes(text):
    text = text.replace("'", "").replace('"', "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["Sentence_sa"] = train_df["Sentence_sa"].apply(normalize_quotes)
dev_df["Sentence_sa"] = dev_df["Sentence_sa"].apply(normalize_quotes)
test_df["Sentence_sa"] = test_df["Sentence_sa"].apply(normalize_quotes)

# Tokenization

In [ ]:
from huggingface_hub import login
login(HF_TOKEN)

print("Logged in to Hugging Face Hub.")

In [59]:
#from huggingface_hub import notebook_login
#notebook_login()

In [ ]:
# Using pretrained tokenizer from IndicTrans2 for tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Tokenizer loaded successfully.")
print("Vocabulary Size :", tokenizer.vocab_size)
print("Pad Token :", tokenizer.pad_token)
print("EOS Token :", tokenizer.eos_token)
print("BOS Token :", tokenizer.bos_token)


from IndicTransToolkit import IndicProcessor
import IndicTransToolkit

ip = IndicProcessor(inference=False)
def preprocess_batch(source_texts, target_texts=None):

    # Preprocess Sanskrit source
    source_texts = ip.preprocess_batch(batch=source_texts, src_lang="san_Deva", tgt_lang="eng_Latn")
    model_inputs = tokenizer(source_texts, max_length=max_source_length, truncation=True, padding="max_length")

    if target_texts is not None:
        # Preprocess English targets
        target_texts = ip.preprocess_batch( batch=target_texts, src_lang="eng_Latn", is_target=True)
        labels = tokenizer(text_target=target_texts, max_length=max_target_length, truncation=True, padding="max_length")

        model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [61]:
from torch.utils.data import Dataset
import torch

class TranslationDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        self.encodings = preprocess_batch(
            self.df["Sentence_sa"].tolist(),
            self.df["Sentence_en"].tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        labels = torch.tensor(
            self.encodings["labels"][idx],
            dtype=torch.long)

        # Ignore padding tokens in the loss
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx],dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx],dtype=torch.long),
            "labels": labels}


from torch.utils.data import DataLoader

train_dataset = TranslationDataset(train_df)
dev_dataset = TranslationDataset(dev_df)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

# Loading IndicTrans2 Model



In [ ]:
# Loading the pretrained IndicTrans2 Indic→English Distilled 200M model
from transformers import AutoConfig, AutoModelForSeq2SeqLM

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.dropout = 0.15
config.attention_dropout = 0.1

model = AutoModelForSeq2SeqLM.from_pretrained(model_name, config=config, trust_remote_code=True)
model.to(device)
print("Model loaded successfully.")


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters : {total:,}")
    print(f"Trainable Parameters : {trainable:,}")

    return total, trainable

total_params, trainable_params = count_parameters(model)

In [63]:
# Optimizer
from torch.optim import AdamW
no_decay = ["bias", "LayerNorm.weight"]

optimizer_grouped_parameters = [{
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": weight_decay, },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,},]

optimizer = AdamW(optimizer_grouped_parameters,lr=learning_rate)

In [ ]:
import math
total_training_steps = (math.ceil(len(train_loader) / gradient_accumulation_steps)* epochs)

use_fp16 = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)
print("Mixed Precision :", use_fp16)

In [ ]:
# Centralized beam-search config
generation_kwargs = {"max_length":max_target_length, "num_beams":num_beams, "length_penalty":0.8, "no_repeat_ngram_size":3, "repetition_penalty":1.1, "early_stopping":True}

batch = next(iter(train_loader))
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)
print("Initial Loss :", outputs.loss.item())

In [66]:
# translate_batch/clean_prediction now defined before the training loop so we can compute dev BLEU each epoch for checkpoint selection
import re
def clean_prediction(text):
  text = re.sub(r"</?s>", "", text)
  text = text.replace("<pad>", "")
  text = text.replace("eng_Latn", "")
  text = text.replace("san_Deva", "")
  text = re.sub(r"\s+", " ", text)
  text = re.sub(r"\s+([.,!?;:])", r"\1", text)
  return text.strip()


@torch.no_grad()
def translate_batch(texts, batch_size=16, gen_kwargs=None):
  gen_kwargs = gen_kwargs or generation_kwargs
  model.eval()
  predictions = []

  for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    processed = ip.preprocess_batch(batch=batch, src_lang="san_Deva", tgt_lang="eng_Latn")
    inputs = tokenizer(processed, return_tensors="pt", padding=True, truncation=True,max_length=max_source_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, **gen_kwargs)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)
    decoded = [clean_prediction(text) for text in decoded]
    predictions.extend(decoded)
  return predictions


def translate_batch_sorted(texts, **kwargs):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    sorted_texts = [texts[i] for i in order]
    sorted_preds = translate_batch(sorted_texts, **kwargs)
    out = [None] * len(texts)
    for orig_i, pred in zip(order, sorted_preds):
        out[orig_i] = pred
    return out

# Training/ Fine-tuning using PyTorch

In [67]:
def train_one_epoch(model, dataloader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=use_fp16):
            outputs = model(**batch)
            loss = outputs.loss
            # Normalize loss
            loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

        if ((step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader)):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.set_postfix(loss=f"{loss.item() * gradient_accumulation_steps:.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def validate(model, dataloader):
    model.eval()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Validation", leave=False)

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import copy
from transformers import get_linear_schedule_with_warmup

best_bleu = -1.0
train_losses = []
val_losses = []
val_bleus = []
patience = 2
early_stop_counter = 0

# Define the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=int(warmup_ratio * total_training_steps),
    num_training_steps=total_training_steps)

best_weights = None
for epoch in range(epochs):
    print("---------------------------")
    print(f"Epoch {epoch+1}/{epochs}")

    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
    val_loss = validate(model, dev_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

    dev_preds_epoch = translate_batch_sorted(dev_df["Sentence_sa"].tolist(), batch_size=16)
    dev_bleu_epoch = nltk_bleu(dev_preds_epoch, dev_df["Sentence_en"].tolist())
    val_bleus.append(dev_bleu_epoch)
    print(f"Dev BLEU: {dev_bleu_epoch:.4f}")

    if dev_bleu_epoch > best_bleu:
        best_bleu = dev_bleu_epoch
        early_stop_counter = 0
        print("Saving best model...\n")

        # Save best weights
        best_weights = copy.deepcopy(model.state_dict())
        # Save checkpoints
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
    else:
        early_stop_counter += 1
        print(f"No improvement ({early_stop_counter}/{patience})")
        if early_stop_counter >= patience:
            print("Early stopping...\n")
            break

# Restore best model before evaluation
if best_weights is not None:
    model.load_state_dict(best_weights)
print(f"Best model restored. Best Dev BLEU = {best_bleu:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Training")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(val_bleus, label="Dev BLEU", color="green")
plt.xlabel("Epoch")
plt.ylabel("BLEU")
plt.title("Dev BLEU per Epoch")
plt.legend()
plt.show()

# Evaluation

In [ ]:
# Load the fine-tuned best model
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, config=config, trust_remote_code=True)

state_dict = load_file(f"{output_dir}/model.safetensors")
model.tie_weights()
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print("best model from disk loaded — training skips and continue for evaluation.")

In [ ]:
# Inference Time for Dev data
import time

start_time = time.time()
dev_predictions = translate_batch_sorted(dev_df["Sentence_sa"].tolist(), batch_size=32)
dev_references = dev_df["Sentence_en"].tolist()
dev_inference_time = time.time() - start_time
print(f"Inference Time (Dev): {dev_inference_time:.2f} sec")

# Inference Time for Test data
start_time2 = time.time()
test_predictions = translate_batch_sorted(test_df["Sentence_sa"].tolist(), batch_size=32)
test_references = test_df["Sentence_en"].tolist()
test_inference_time = time.time() - start_time2
print(f"Inference Time (Test) : {test_inference_time:.2f} sec")

In [ ]:
# nltk BLEU
final_nltk_dev_bleu = nltk_bleu(dev_predictions, dev_references)
print(f"NLTK BLEU Score (Dev) : {final_nltk_dev_bleu:.4f}")

final_nltk_test_bleu = nltk_bleu(test_predictions, test_references)
print(f"NLTK BLEU Score (Test) : {final_nltk_test_bleu:.4f}")

In [ ]:
# hf evalualte BLEU
hf_dev_bleu = evaluate_bleu_hf(dev_predictions, dev_references)
print(f"HF evaluate BLEU (Dev) : {hf_dev_bleu['bleu']:.4f}")

hf_test_bleu = evaluate_bleu_hf(test_predictions, test_references)
print(f"HF evaluate BLEU (Test): {hf_test_bleu['bleu']:.4f}")

In [ ]:
# BERT Score
from bert_score import score as bertscore
P, R, F1 = bertscore(dev_predictions, dev_references, lang="en", rescale_with_baseline=True)
print(f"BERTScore (F1) Dev: {F1.mean().item():.4f}")

lowest = torch.argsort(F1)[:20]

for idx in lowest:
    i = idx.item()
    print("------------------------------")
    print("F1 :", F1[i].item())
    print("Source :", dev_df.iloc[i]["Sentence_sa"])
    print("Reference :", dev_references[i])
    print("Prediction :", dev_predictions[i])

In [ ]:
P_test, R_test, F1_test = bertscore(test_predictions, test_references, lang="en", rescale_with_baseline=True)
print(f"BERTScore (F1) Test: {F1_test.mean().item():.4f}")

In [ ]:
sample_results = pd.DataFrame({
    "Source": dev_df["Sentence_sa"][:10],
    "Reference": dev_references[:10],
    "Prediction": dev_predictions[:10]})

sample_results

In [ ]:
submission = pd.DataFrame({"Source_id": test_df["Source_id"], "Sentence_en": test_predictions})
submission.to_csv("/content/drive/MyDrive/submission.csv", index=False, encoding="utf-8")
print("File saved successfully.")

In [ ]:
print(f"NLTK BLEU (Dev): {final_nltk_dev_bleu:.4f}")
print(f"HF BLEU (Dev): {hf_dev_bleu['bleu']:.4f}")
print(f"BERT Score (F1) Dev: {F1.mean().item():.4f}\n")
print(f"NLTK BLEU(Test): {final_nltk_test_bleu:.4f}")
print(f"HF BLEU (Test): {hf_test_bleu['bleu']:.4f}")
print(f"BERT Score (F1) Test: {F1_test.mean().item():.4f}\n")
print(f"Inference Time (Dev) : {dev_inference_time:.2f} sec")
print(f"Inference Time (Test) : {test_inference_time:.2f} sec\n")
print(f"Total Parameters : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

In [ ]:
# Predict from new test dataset
'''new_test_df = pd.read_csv('/content/drive/MyDrive/Dataset_NLU/test_sa_1000.csv')
new_test_predictions = translate_batch_sorted(new_test_df["Sentence_sa"].tolist(), batch_size=16)
submission.to_csv("/content/drive/MyDrive/new_submission.csv", index=False, encoding="utf-8")
print("Sample result file saved successfully.")'''